In [1]:
# importações, caminhos, categorias e funções auxiliares
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training"
EVALUATION_RESULTS_DIR = ROOT / "results" / "model_evaluation"
ANALYSIS_DIR = ROOT / "results" / "results_analysis"
TABLES_DIR = ANALYSIS_DIR / "tables"
BASELINE_TABLES_DIR = TABLES_DIR / "baseline"
RANDOMIZED_TABLES_DIR = TABLES_DIR / "randomized_search"
COMPARISON_TABLES_DIR = TABLES_DIR / "comparisons"
FIGURES_DIR = ANALYSIS_DIR / "figures"

for output_directory in [BASELINE_TABLES_DIR, RANDOMIZED_TABLES_DIR, COMPARISON_TABLES_DIR, FIGURES_DIR]:
    output_directory.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
METHOD_ORDER = ["baseline", "randomized_search"]
SCENARIO_ORDER = ["intra_dataset", "cross_dataset", "combined_to_unseen"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
TEST_SOURCE_ORDER = ["cicids2017", "unsw_nb15", "iot23"]

MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
METHOD_LABELS = {"baseline": "Baseline", "randomized_search": "RandomizedSearchCV"}
SCENARIO_LABELS = {
    "intra_dataset": "Intra-dataset",
    "cross_dataset": "Cross-dataset",
    "combined_to_unseen": "Combinação para base não vista",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
METHOD_COLORS = {"Baseline": "#4C78A8", "RandomizedSearchCV": "#F58518"}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


def save_table(dataframe, output_directory, filename):
    # os CSVs preservam os valores completos e serão formatados somente nas tabelas finais
    output_path = output_directory / filename
    dataframe.to_csv(output_path, index=False)
    return output_path


def finalize_figure(figure, filename, width, height):
    # o gráfico permanece interativo no notebook e é exportado somente em SVG
    figure.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=13),
        title_x=0.5,
        margin=dict(l=80, r=40, t=130, b=80),
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
    )
    output_path = FIGURES_DIR / f"{filename}.svg"
    figure.write_image(output_path, format="svg", width=width, height=height)
    display(figure)
    return output_path

In [2]:
# carregamento dos resultados e validação da integridade dos registros
baseline_summary = pd.read_csv(TRAINING_RESULTS_DIR / "baseline" / "summary.csv")
randomized_summary_files = [
    TRAINING_RESULTS_DIR / "randomized_search" / model_name / "summary.csv"
    for model_name in MODEL_ORDER
]
randomized_summary = pd.concat(
    [pd.read_csv(summary_path) for summary_path in randomized_summary_files],
    ignore_index=True,
)
randomized_summary["training_method"] = "randomized_search"
evaluation = pd.read_csv(EVALUATION_RESULTS_DIR / "evaluation_summary.csv")
evaluation.insert(0, "evaluation_order", np.arange(1, len(evaluation) + 1))
comparison = pd.read_csv(EVALUATION_RESULTS_DIR / "baseline_vs_randomized_search.csv")
cv_results = pd.concat([baseline_summary, randomized_summary], ignore_index=True, sort=False)

assert len(baseline_summary) == 30, "O summary do baseline deve possuir 30 registros."
assert len(randomized_summary) == 30, "Os summaries do RandomizedSearch devem totalizar 30 registros."
assert len(evaluation) == 120, "A avaliação final deve possuir 120 registros."
assert len(comparison) == 60, "A comparação deve possuir 60 pares."
assert not cv_results.duplicated(["training_method", "model", "training_source"]).any()
assert not evaluation.duplicated(["training_method", "model", "training_source", "test_source"]).any()
assert not comparison.duplicated(["model", "training_source", "test_source"]).any()

required_metrics = ["accuracy", "balanced_accuracy", "precision_attack", "recall_attack", "f1_attack", "f1_macro"]
assert not evaluation[required_metrics].isna().any().any(), "Há métricas ausentes."
for metric in required_metrics:
    assert evaluation[metric].between(0, 1).all(), f"Valores inválidos em {metric}."

validation_summary = pd.DataFrame({
    "item": ["summary baseline", "summaries RandomizedSearch", "avaliações finais", "comparações pareadas"],
    "registros": [len(baseline_summary), len(randomized_summary), len(evaluation), len(comparison)],
    "status": "ok",
})
display(validation_summary)

,item,registros,status
0,summary baseline,30,ok
1,summaries RandomizedSearch,30,ok
2,avaliações finais,120,ok
3,comparações pareadas,60,ok


In [3]:
# tabelas completas separadas por método e por tipo de cenário
cv_table_columns = [
    "model", "training_method", "training_source", "training_rows", "n_splits", "params",
    "precision_attack_mean", "precision_attack_std", "recall_attack_mean", "recall_attack_std",
    "f1_attack_mean", "f1_attack_std", "f1_macro_mean", "f1_macro_std",
    "accuracy_mean", "accuracy_std", "balanced_accuracy_mean", "balanced_accuracy_std",
    "elapsed_seconds", "elapsed_minutes", "refit_time_seconds", "model_size_mb", "model_file",
]
final_table_columns = [
    "evaluation_order", "training_method", "training_source", "test_source", "model",
    "scenario_type", "params", "test_rows", "benign_support", "attack_support",
    "accuracy", "balanced_accuracy", "precision_attack", "recall_attack", "f1_attack",
    "precision_benign", "recall_benign", "f1_benign", "f1_macro", "f1_weighted",
    "true_negative", "false_positive", "false_negative", "true_positive",
    "cv_n_splits", "cv_accuracy_mean", "cv_accuracy_std",
    "cv_balanced_accuracy_mean", "cv_balanced_accuracy_std",
    "cv_precision_attack_mean", "cv_precision_attack_std",
    "cv_recall_attack_mean", "cv_recall_attack_std",
    "cv_f1_attack_mean", "cv_f1_attack_std",
    "cv_f1_macro_mean", "cv_f1_macro_std",
    "prediction_time_seconds", "prediction_rows_per_second", "metrics_calculation_time_seconds",
    "training_rows", "training_elapsed_seconds", "training_elapsed_minutes",
    "final_fit_time_seconds", "model_size_mb",
    "classification_report_file", "confusion_matrix_file",
]
comparison_table_columns = [
    "model", "training_source", "test_source", "scenario_type",
    "f1_attack_baseline", "f1_attack_randomized_search", "f1_attack_delta",
    "balanced_accuracy_baseline", "balanced_accuracy_randomized_search", "balanced_accuracy_delta",
    "precision_attack_baseline", "precision_attack_randomized_search", "precision_attack_delta",
    "recall_attack_baseline", "recall_attack_randomized_search", "recall_attack_delta",
    "cv_f1_attack_mean_baseline", "cv_f1_attack_std_baseline",
    "cv_f1_attack_mean_randomized_search", "cv_f1_attack_std_randomized_search",
    "cv_balanced_accuracy_mean_baseline", "cv_balanced_accuracy_std_baseline",
    "cv_balanced_accuracy_mean_randomized_search", "cv_balanced_accuracy_std_randomized_search",
    "params_randomized_search",
]
METHOD_TABLE_DIRECTORIES = {"baseline": BASELINE_TABLES_DIR, "randomized_search": RANDOMIZED_TABLES_DIR}
saved_tables = []
scenario_tables = {}

for method, output_directory in METHOD_TABLE_DIRECTORIES.items():
    method_cv = (
        cv_results[cv_results["training_method"].eq(method)]
        .reindex(columns=cv_table_columns)
        .sort_values(["training_source", "model"])
    )
    method_evaluation = evaluation[
        evaluation["training_method"].eq(method)
    ][final_table_columns].copy()
    best_by_scenario = (
        method_evaluation.sort_values(["f1_attack", "balanced_accuracy"], ascending=[False, False])
        .groupby(["training_source", "test_source"], as_index=False)
        .head(1)
        .sort_values(["training_source", "test_source"])
    )
    training_cost = (
        method_evaluation[[
            "model", "training_method", "training_source", "training_elapsed_seconds",
            "final_fit_time_seconds", "model_size_mb", "params",
        ]]
        .drop_duplicates(["model", "training_source"])
        .sort_values(["model", "training_source"])
    )
    prediction = method_evaluation[[
        "model", "training_method", "training_source", "test_source", "scenario_type",
        "prediction_time_seconds", "prediction_rows_per_second", "metrics_calculation_time_seconds",
    ]]
    saved_tables.extend([
        save_table(method_cv, output_directory, "kfold_metrics.csv"),
        save_table(method_evaluation, output_directory, "final_metrics_all_scenarios.csv"),
        save_table(best_by_scenario, output_directory, "best_model_by_scenario.csv"),
        save_table(training_cost, output_directory, "training_cost_and_model_size.csv"),
        save_table(prediction, output_directory, "prediction_time_and_throughput.csv"),
    ])
    for scenario_type in SCENARIO_ORDER:
        scenario_table = method_evaluation[
            method_evaluation["scenario_type"].eq(scenario_type)
        ].copy()
        scenario_tables[(method, scenario_type)] = scenario_table
        saved_tables.append(save_table(
            scenario_table, output_directory, f"scenario_{scenario_type}_all_experiments.csv"
        ))

comparison_metrics_table = comparison[comparison_table_columns].sort_values(
    ["scenario_type", "training_source", "test_source", "model"]
)
saved_tables.append(save_table(
    comparison_metrics_table,
    COMPARISON_TABLES_DIR,
    "baseline_vs_randomized_search_all_scenarios.csv",
))
for scenario_type in SCENARIO_ORDER:
    scenario_comparison = comparison_metrics_table[comparison_metrics_table["scenario_type"].eq(scenario_type)]
    saved_tables.append(save_table(
        scenario_comparison,
        COMPARISON_TABLES_DIR,
        f"baseline_vs_randomized_search_{scenario_type}.csv",
    ))

display(pd.DataFrame({
    "tabela_salva": [path.relative_to(TABLES_DIR).as_posix() for path in saved_tables]
}))

# exibição das seis tabelas completas por método e cenário
for method in METHOD_ORDER:
    print(f"\n{METHOD_LABELS[method]}")
    for scenario_type in SCENARIO_ORDER:
        print(f"\n{SCENARIO_LABELS[scenario_type]}")
        display(scenario_tables[(method, scenario_type)])

,tabela_salva
0,baseline/kfold_metrics.csv
1,baseline/final_metrics_all_scenarios.csv
2,baseline/best_model_by_scenario.csv
3,baseline/training_cost_and_model_size.csv
4,baseline/prediction_time_and_throughput.csv
5,baseline/scenario_intra_dataset_all_experiments.csv
6,baseline/scenario_cross_dataset_all_experiments.csv
7,baseline/scenario_combined_to_unseen_all_experiments.csv
8,randomized_search/kfold_metrics.csv
9,randomized_search/final_metrics_all_scenarios.csv



Baseline

Intra-dataset


,evaluation_order,training_method,training_source,test_source,model,scenario_type,params,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,f1_weighted,true_negative,false_positive,false_negative,true_positive,cv_n_splits,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_attack_mean,cv_precision_attack_std,cv_recall_attack_mean,cv_recall_attack_std,cv_f1_attack_mean,cv_f1_attack_std,cv_f1_macro_mean,cv_f1_macro_std,prediction_time_seconds,prediction_rows_per_second,metrics_calculation_time_seconds,training_rows,training_elapsed_seconds,training_elapsed_minutes,final_fit_time_seconds,model_size_mb,classification_report_file,confusion_matrix_file
0,1,baseline,cicids2017,cicids2017,decision_tree,intra_dataset,{},100925,80740,20185,0.994055,0.992123,0.981512,0.988903,0.985193,0.997220,0.995343,0.996281,0.990737,0.994063,80364,376,224,19961,5,0.993817,0.000232,0.991478,0.000652,0.981621,0.001258,0.987579,0.001500,0.984589,0.000582,0.990361,0.000363,0.025293,3.990156e+06,0.047184,235490,6.396218,0.106604,1.682596,0.033245,results\model_evaluation\cicids2017\decision_tree\baseline_cicids2017.txt,results\model_evaluation\cicids2017\decision_tree\baseline_cicids2017_confusion_matrix.txt
2,3,baseline,cicids2017,cicids2017,linear_svm,intra_dataset,{},100925,80740,20185,0.888135,0.763197,0.829225,0.554967,0.664926,0.897238,0.971427,0.932860,0.798893,0.879273,78433,2307,8983,11202,5,0.887706,0.000912,0.761975,0.001436,0.829097,0.004660,0.552423,0.002746,0.663048,0.002611,0.797837,0.001565,0.024238,4.163985e+06,0.055290,235490,14.703877,0.245065,7.384577,0.001984,results\model_evaluation\cicids2017\linear_svm\baseline_cicids2017.txt,results\model_evaluation\cicids2017\linear_svm\baseline_cicids2017_confusion_matrix.txt
4,5,baseline,cicids2017,cicids2017,mlp,intra_dataset,{},100925,80740,20185,0.993678,0.992909,0.977105,0.991627,0.984313,0.997899,0.994191,0.996042,0.990177,0.993696,80271,469,169,20016,5,0.992840,0.000799,0.990461,0.002442,0.977916,0.002577,0.986496,0.005328,0.982174,0.002037,0.988847,0.001267,0.051851,1.946431e+06,0.075223,235490,83.291203,1.388187,31.641816,0.059023,results\model_evaluation\cicids2017\mlp\baseline_cicids2017.txt,results\model_evaluation\cicids2017\mlp\baseline_cicids2017_confusion_matrix.txt
6,7,baseline,cicids2017,cicids2017,random_forest,intra_dataset,{},100925,80740,20185,0.994848,0.994309,0.981066,0.993411,0.987200,0.998348,0.995207,0.996775,0.991987,0.994860,80353,387,133,20052,5,0.994900,0.000300,0.994137,0.000566,0.981841,0.001440,0.992866,0.001230,0.987321,0.000744,0.992065,0.000466,0.442691,2.279809e+05,0.049349,235490,49.531468,0.825524,22.162800,3.159244,results\model_evaluation\cicids2017\random_forest\baseline_cicids2017.txt,results\model_evaluation\cicids2017\random_forest\baseline_cicids2017_confusion_matrix.txt
8,9,baseline,cicids2017,cicids2017,xgboost,intra_dataset,{},100925,80740,20185,0.994580,0.994346,0.979256,0.993956,0.986551,0.998483,0.994736,0.996606,0.991579,0.994595,80315,425,122,20063,5,0.994696,0.000332,0.994042,0.000669,0.980771,0.001223,0.992951,0.001367,0.986822,0.000826,0.991751,0.000517,0.206462,4.888318e+05,0.058120,235490,4.028532,0.067142,1.565781,0.093504,results\model_evaluation\cicids2017\xgboost\baseline_cicids2017.txt,results\model_evaluation\cicids2017\xgboost\baseline_cicids2017_confusion_matrix.txt
60,61,baseline,iot23,iot23,decision_tree,intra_dataset,{},100925,80740,20185,0.998682,0.997207,0.998657,0.994749,0.996699,0.998688,0.999666,0.999177,0.997938,0.998681,80713,27,106,20079,5,0.998505,0.000199,0.996685,0.000457,0.998869,0.000220,0.993652,0.000892,0.996253,0.000499,0.997660,0.000312,0.026839,3.760414e+06,0.057911,235490,3.376987,0.056283,0.569364,0.010662,results\model_evaluation\iot23\decision_tree\baseline_iot23.txt,results\model_evaluation\iot23\decision_tree\baseline_iot23_confusion_matrix.txt
62,63,ba


Cross-dataset


,evaluation_order,training_method,training_source,test_source,model,scenario_type,params,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,f1_weighted,true_negative,false_positive,false_negative,true_positive,cv_n_splits,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_attack_mean,cv_precision_attack_std,cv_recall_attack_mean,cv_recall_attack_std,cv_f1_attack_mean,cv_f1_attack_std,cv_f1_macro_mean,cv_f1_macro_std,prediction_time_seconds,prediction_rows_per_second,metrics_calculation_time_seconds,training_rows,training_elapsed_seconds,training_elapsed_minutes,final_fit_time_seconds,model_size_mb,classification_report_file,confusion_matrix_file
10,11,baseline,cicids2017,iot23,decision_tree,cross_dataset,{},100925,80740,20185,0.804865,0.523105,0.647094,0.053505,0.098838,0.807518,0.992705,0.890586,0.494712,0.732237,80151,589,19105,1080,5,0.993817,0.000232,0.991478,0.000652,0.981621,0.001258,0.987579,0.001500,0.984589,0.000582,0.990361,0.000363,0.023891,4.224412e+06,0.045886,235490,6.396218,0.106604,1.682596,0.033245,results\model_evaluation\cicids2017\decision_tree\baseline_iot23.txt,results\model_evaluation\cicids2017\decision_tree\baseline_iot23_confusion_matrix.txt
12,13,baseline,cicids2017,iot23,linear_svm,cross_dataset,{},100925,80740,20185,0.110964,0.107809,0.028094,0.102551,0.044105,0.335083,0.113067,0.169081,0.106593,0.144086,9129,71611,18115,2070,5,0.887706,0.000912,0.761975,0.001436,0.829097,0.004660,0.552423,0.002746,0.663048,0.002611,0.797837,0.001565,0.019316,5.224862e+06,0.048588,235490,14.703877,0.245065,7.384577,0.001984,results\model_evaluation\cicids2017\linear_svm\baseline_iot23.txt,results\model_evaluation\cicids2017\linear_svm\baseline_iot23_confusion_matrix.txt
14,15,baseline,cicids2017,iot23,mlp,cross_dataset,{},100925,80740,20185,0.251117,0.460887,0.185664,0.810503,0.302121,0.701382,0.111271,0.192070,0.247096,0.214081,8984,71756,3825,16360,5,0.992840,0.000799,0.990461,0.002442,0.977916,0.002577,0.986496,0.005328,0.982174,0.002037,0.988847,0.001267,0.068127,1.481414e+06,0.062555,235490,83.291203,1.388187,31.641816,0.059023,results\model_evaluation\cicids2017\mlp\baseline_iot23.txt,results\model_evaluation\cicids2017\mlp\baseline_iot23_confusion_matrix.txt
16,17,baseline,cicids2017,iot23,random_forest,cross_dataset,{},100925,80740,20185,0.099846,0.064671,0.001721,0.006044,0.002679,0.331634,0.123297,0.179761,0.091220,0.144345,9955,70785,20063,122,5,0.994900,0.000300,0.994137,0.000566,0.981841,0.001440,0.992866,0.001230,0.987321,0.000744,0.992065,0.000466,0.311708,3.237809e+05,0.068568,235490,49.531468,0.825524,22.162800,3.159244,results\model_evaluation\cicids2017\random_forest\baseline_iot23.txt,results\model_evaluation\cicids2017\random_forest\baseline_iot23_confusion_matrix.txt
18,19,baseline,cicids2017,iot23,xgboost,cross_dataset,{},100925,80740,20185,0.094922,0.063228,0.002934,0.010404,0.004577,0.319305,0.116052,0.170232,0.087405,0.137101,9370,71370,19975,210,5,0.994696,0.000332,0.994042,0.000669,0.980771,0.001223,0.992951,0.001367,0.986822,0.000826,0.991751,0.000517,0.140193,7.199020e+05,0.048996,235490,4.028532,0.067142,1.565781,0.093504,results\model_evaluation\cicids2017\xgboost\baseline_iot23.txt,results\model_evaluation\cicids2017\xgboost\baseline_iot23_confusion_matrix.txt
20,21,baseline,cicids2017,unsw_nb15,decision_tree,cross_dataset,{},100925,80740,20185,0.777221,0.486859,0.024410,0.002923,0.005221,0.795692,0.970795,0.874565,0.439893,0.700696,78382,2358,20126,59,5,0.993817,0.000232,0.991478,0.000652,0.981621,0.001258,0.987579,0.001500,0.984589,0.000582,0.990361,0.000363,0.024148,4.179435e+06,0.046580,235490,6.396218,0.106604,1.682596,0.033245,results\model_evaluation\cicids2017\decision_tree\baseline_unsw_nb15.txt,results\model_evaluation\cicids2017\decision_tree\baseline_unsw_nb15_confusion_matrix.txt
22,23,baseline,cicids2017,unsw_nb15,line

...

,evaluation_order,training_method,training_source,test_source,model,scenario_type,params,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,f1_weighted,true_negative,false_positive,false_negative,true_positive,cv_n_splits,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_attack_mean,cv_precision_attack_std,cv_recall_attack_mean,cv_recall_attack_std,cv_f1_attack_mean,cv_f1_attack_std,cv_f1_macro_mean,cv_f1_macro_std,prediction_time_seconds,prediction_rows_per_second,metrics_calculation_time_seconds,training_rows,training_elapsed_seconds,training_elapsed_minutes,final_fit_time_seconds,model_size_mb,classification_report_file,confusion_matrix_file
1,2,randomized_search,cicids2017,cicids2017,decision_tree,intra_dataset,"{""classifier__class_weight"": null, ""classifier__max_depth"": 15, ""classifier__min_samples_leaf"": 4, ""classifier__min_...",100925,80740,20185,0.993579,0.992104,0.978496,0.989646,0.984039,0.997404,0.994563,0.995981,0.990010,0.993593,80301,439,209,19976,5,0.993698,0.000324,0.991977,0.000383,0.979586,0.001766,0.989108,0.000861,0.984322,0.000794,0.990190,0.000499,0.023318,4.328127e+06,0.047638,235490,19.661492,0.327692,1.380483,0.016391,results\model_evaluation\cicids2017\decision_tree\randomized_search_cicids2017.txt,results\model_evaluation\cicids2017\decision_tree\randomized_search_cicids2017_confusion_matrix.txt
3,4,randomized_search,cicids2017,cicids2017,linear_svm,intra_dataset,"{""classifier__C"": 46.00306804490304, ""classifier__class_weight"": ""balanced""}",100925,80740,20185,0.931504,0.942829,0.759706,0.961704,0.848853,0.989744,0.923953,0.955718,0.902286,0.934345,74600,6140,773,19412,5,0.931916,0.001341,0.943530,0.001734,0.760466,0.003145,0.962886,0.002528,0.849787,0.002813,0.902885,0.001845,0.019476,5.181966e+06,0.051128,235490,55.001995,0.916700,5.579139,0.001999,results\model_evaluation\cicids2017\linear_svm\randomized_search_cicids2017.txt,results\model_evaluation\cicids2017\linear_svm\randomized_search_cicids2017_confusion_matrix.txt
5,6,randomized_search,cicids2017,cicids2017,mlp,intra_dataset,"{""classifier__alpha"": 1.9726066891840947e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"":...",100925,80740,20185,0.993906,0.993702,0.976573,0.993361,0.984896,0.998333,0.994043,0.996183,0.990540,0.993926,80259,481,134,20051,5,0.994051,0.000530,0.993790,0.000533,0.977278,0.002275,0.993354,0.000752,0.985249,0.001300,0.990762,0.000817,0.057927,1.742288e+06,0.046958,235490,230.235576,3.837260,21.293389,0.101001,results\model_evaluation\cicids2017\mlp\randomized_search_cicids2017.txt,results\model_evaluation\cicids2017\mlp\randomized_search_cicids2017_confusion_matrix.txt
7,8,randomized_search,cicids2017,cicids2017,random_forest,intra_dataset,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 1, ""classifi...",100925,80740,20185,0.994818,0.994309,0.980875,0.993460,0.987127,0.998360,0.995157,0.996756,0.991942,0.994830,80349,391,132,20053,5,0.994989,0.000340,0.994416,0.000504,0.981708,0.001679,0.993460,0.001069,0.987548,0.000839,0.992206,0.000526,1.043335,9.673303e+04,0.052324,235490,622.521286,10.375355,47.460043,7.689332,results\model_evaluation\cicids2017\random_forest\randomized_search_cicids2017.txt,results\model_evaluation\cicids2017\random_forest\randomized_search_cicids2017_confusion_matrix.txt
9,10,randomized_search,cicids2017,cicids2017,xgboost,intra_dataset,"{""classifier__colsample_bytree"": 0.717772960390469, ""classifier__learning_rate"": 0.1404912558872892, ""classifier__ma...",100925,80740,20185,0.994570,0.994210,0.979536,0.993609,0.986522,0.998397,0.994811,0.996600,0.991561,0.994585,80321,419,129,20056,5,0.994751,0.000386,0.994100,0.000783,0.980977,0.001133,0.993015,0.001540,0.986958,0.000962,0.991836,0.000601,0.400620,2.519218e+05,0.048434,235490,50.081


Cross-dataset


,evaluation_order,training_method,training_source,test_source,model,scenario_type,params,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,f1_weighted,true_negative,false_positive,false_negative,true_positive,cv_n_splits,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_attack_mean,cv_precision_attack_std,cv_recall_attack_mean,cv_recall_attack_std,cv_f1_attack_mean,cv_f1_attack_std,cv_f1_macro_mean,cv_f1_macro_std,prediction_time_seconds,prediction_rows_per_second,metrics_calculation_time_seconds,training_rows,training_elapsed_seconds,training_elapsed_minutes,final_fit_time_seconds,model_size_mb,classification_report_file,confusion_matrix_file
11,12,randomized_search,cicids2017,iot23,decision_tree,cross_dataset,"{""classifier__class_weight"": null, ""classifier__max_depth"": 15, ""classifier__min_samples_leaf"": 4, ""classifier__min_...",100925,80740,20185,0.103879,0.086271,0.015836,0.056923,0.024779,0.329033,0.115618,0.171110,0.097945,0.141844,9335,71405,19036,1149,5,0.993698,0.000324,0.991977,0.000383,0.979586,0.001766,0.989108,0.000861,0.984322,0.000794,0.990190,0.000499,0.023079,4.373118e+06,0.049546,235490,19.661492,0.327692,1.380483,0.016391,results\model_evaluation\cicids2017\decision_tree\randomized_search_iot23.txt,results\model_evaluation\cicids2017\decision_tree\randomized_search_iot23_confusion_matrix.txt
13,14,randomized_search,cicids2017,iot23,linear_svm,cross_dataset,"{""classifier__C"": 46.00306804490304, ""classifier__class_weight"": ""balanced""}",100925,80740,20185,0.286242,0.552452,0.218400,0.996136,0.358254,0.991196,0.108769,0.196027,0.277140,0.228472,8782,71958,78,20107,5,0.931916,0.001341,0.943530,0.001734,0.760466,0.003145,0.962886,0.002528,0.849787,0.002813,0.902885,0.001845,0.018331,5.505851e+06,0.051706,235490,55.001995,0.916700,5.579139,0.001999,results\model_evaluation\cicids2017\linear_svm\randomized_search_iot23.txt,results\model_evaluation\cicids2017\linear_svm\randomized_search_iot23_confusion_matrix.txt
15,16,randomized_search,cicids2017,iot23,mlp,cross_dataset,"{""classifier__alpha"": 1.9726066891840947e-05, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"":...",100925,80740,20185,0.250671,0.460497,0.185527,0.810206,0.301918,0.700141,0.110788,0.191304,0.246611,0.213427,8945,71795,3831,16354,5,0.994051,0.000530,0.993790,0.000533,0.977278,0.002275,0.993354,0.000752,0.985249,0.001300,0.990762,0.000817,0.052290,1.930116e+06,0.050233,235490,230.235576,3.837260,21.293389,0.101001,results\model_evaluation\cicids2017\mlp\randomized_search_iot23.txt,results\model_evaluation\cicids2017\mlp\randomized_search_iot23_confusion_matrix.txt
17,18,randomized_search,cicids2017,iot23,random_forest,cross_dataset,"{""classifier__class_weight"": ""balanced"", ""classifier__max_depth"": null, ""classifier__min_samples_leaf"": 1, ""classifi...",100925,80740,20185,0.098885,0.064107,0.001746,0.006143,0.002720,0.329445,0.122071,0.178136,0.090428,0.143053,9856,70884,20061,124,5,0.994989,0.000340,0.994416,0.000504,0.981708,0.001679,0.993460,0.001069,0.987548,0.000839,0.992206,0.000526,0.638293,1.581171e+05,0.046784,235490,622.521286,10.375355,47.460043,7.689332,results\model_evaluation\cicids2017\random_forest\randomized_search_iot23.txt,results\model_evaluation\cicids2017\random_forest\randomized_search_iot23_confusion_matrix.txt
19,20,randomized_search,cicids2017,iot23,xgboost,cross_dataset,"{""classifier__colsample_bytree"": 0.717772960390469, ""classifier__learning_rate"": 0.1404912558872892, ""classifier__ma...",100925,80740,20185,0.789814,0.495287,0.073798,0.004409,0.008321,0.798474,0.986165,0.882450,0.445386,0.707624,79623,1117,20096,89,5,0.994751,0.000386,0.994100,0.000783,0.980977,0.001133,0.993015,0.001540,0.986958,0.000962,0.991836,0.000601,0.377479,2.673659e+05,0.052444,235490,50.081530,0.834692,6.043281,0.256427,results\model_evaluat


Combinação para base não vista


,evaluation_order,training_method,training_source,test_source,model,scenario_type,params,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro,f1_weighted,true_negative,false_positive,false_negative,true_positive,cv_n_splits,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_attack_mean,cv_precision_attack_std,cv_recall_attack_mean,cv_recall_attack_std,cv_f1_attack_mean,cv_f1_attack_std,cv_f1_macro_mean,cv_f1_macro_std,prediction_time_seconds,prediction_rows_per_second,metrics_calculation_time_seconds,training_rows,training_elapsed_seconds,training_elapsed_minutes,final_fit_time_seconds,model_size_mb,classification_report_file,confusion_matrix_file
31,32,randomized_search,cicids2017_iot23,unsw_nb15,decision_tree,combined_to_unseen,"{""classifier__class_weight"": null, ""classifier__max_depth"": 20, ""classifier__min_samples_leaf"": 10, ""classifier__min...",100925,80740,20185,0.803716,0.518801,0.634024,0.043944,0.082191,0.806101,0.993659,0.890107,0.486149,0.728524,80228,512,19298,887,5,0.995717,0.000249,0.993748,0.000350,0.988149,0.000841,0.990467,0.000564,0.989306,0.000620,0.993314,0.000388,0.028968,3.483969e+06,0.048136,470980,30.642596,0.510710,2.950768,0.024882,results\model_evaluation\cicids2017_iot23\decision_tree\randomized_search_unsw_nb15.txt,results\model_evaluation\cicids2017_iot23\decision_tree\randomized_search_unsw_nb15_confusion_matrix.txt
33,34,randomized_search,cicids2017_iot23,unsw_nb15,linear_svm,combined_to_unseen,"{""classifier__C"": 0.4944059287398678, ""classifier__class_weight"": null}",100925,80740,20185,0.786584,0.528678,0.373316,0.098836,0.156293,0.809690,0.958521,0.877842,0.517067,0.733532,77391,3349,18190,1995,5,0.965372,0.001083,0.958488,0.001420,0.887418,0.003417,0.947015,0.002213,0.916246,0.002542,0.947210,0.001615,0.019010,5.308936e+06,0.048142,470980,142.201567,2.370026,22.302443,0.001997,results\model_evaluation\cicids2017_iot23\linear_svm\randomized_search_unsw_nb15.txt,results\model_evaluation\cicids2017_iot23\linear_svm\randomized_search_unsw_nb15_confusion_matrix.txt
35,36,randomized_search,cicids2017_iot23,unsw_nb15,mlp,combined_to_unseen,"{""classifier__alpha"": 0.001026603003516424, ""classifier__early_stopping"": false, ""classifier__hidden_layer_sizes"": [...",100925,80740,20185,0.806104,0.519941,0.775000,0.043002,0.081483,0.806453,0.996879,0.891612,0.486547,0.729586,80488,252,19317,868,5,0.995422,0.000248,0.993245,0.000646,0.987521,0.000367,0.989617,0.001323,0.988568,0.000627,0.992853,0.000391,0.071212,1.417243e+06,0.047352,470980,498.192450,8.303208,145.423687,0.175910,results\model_evaluation\cicids2017_iot23\mlp\randomized_search_unsw_nb15.txt,results\model_evaluation\cicids2017_iot23\mlp\randomized_search_unsw_nb15_confusion_matrix.txt
37,38,randomized_search,cicids2017_iot23,unsw_nb15,random_forest,combined_to_unseen,"{""classifier__class_weight"": null, ""classifier__max_depth"": 30, ""classifier__min_samples_leaf"": 6, ""classifier__n_es...",100925,80740,20185,0.804687,0.513723,0.843251,0.028784,0.055667,0.804422,0.998662,0.891079,0.473373,0.723997,80632,108,19604,581,5,0.996437,0.000199,0.995293,0.000351,0.988851,0.000587,0.993386,0.000645,0.991113,0.000497,0.994443,0.000311,1.008007,1.001233e+05,0.048602,470980,1343.521462,22.392024,119.275517,6.447956,results\model_evaluation\cicids2017_iot23\random_forest\randomized_search_unsw_nb15.txt,results\model_evaluation\cicids2017_iot23\random_forest\randomized_search_unsw_nb15_confusion_matrix.txt
39,40,randomized_search,cicids2017_iot23,unsw_nb15,xgboost,combined_to_unseen,"{""classifier__colsample_bytree"": 0.717772960390469, ""classifier__learning_rate"": 0.1404912558872892, ""classifier__ma...",100925,80740,20185,0.799832,0.500062,0.257143,0.000446,0.000890,0.800020,0.999678,0.888774,0.444832,0.711197,80714,26,20176,9,5,0.996263,0.000259,0.994890,0.000448,0.988759,0.000703,0.992

In [4]:
# seleção transparente de casos candidatos para discussão posterior
comparison_case_columns = [
    "model", "training_source", "test_source", "scenario_type",
    "f1_attack_baseline", "f1_attack_randomized_search", "f1_attack_delta",
    "balanced_accuracy_delta", "precision_attack_delta", "recall_attack_delta",
]


def label_comparison_cases(rows, case_type, selection_rule):
    output = rows[comparison_case_columns].copy()
    output.insert(0, "selection_rule", selection_rule)
    output.insert(0, "case_type", case_type)
    return output


comparison_with_abs_delta = comparison.assign(f1_attack_abs_delta=comparison["f1_attack_delta"].abs())
comparison_cases = pd.concat([
    label_comparison_cases(
        comparison.nlargest(3, "f1_attack_delta"),
        "maior_melhoria_f1_ataque",
        "três maiores valores de f1_attack_delta",
    ),
    label_comparison_cases(
        comparison.nsmallest(3, "f1_attack_delta"),
        "maior_piora_f1_ataque",
        "três menores valores de f1_attack_delta",
    ),
    label_comparison_cases(
        comparison_with_abs_delta.nsmallest(3, "f1_attack_abs_delta"),
        "menor_alteracao_f1_ataque",
        "três menores variações absolutas de f1_attack_delta",
    ),
    label_comparison_cases(
        comparison.nlargest(3, "precision_attack_delta"),
        "maior_melhoria_precisao_ataque",
        "três maiores valores de precision_attack_delta",
    ),
    label_comparison_cases(
        comparison.nlargest(3, "recall_attack_delta"),
        "maior_melhoria_recall_ataque",
        "três maiores valores de recall_attack_delta",
    ),
], ignore_index=True)

performance_case_columns = [
    "model", "training_method", "training_source", "test_source", "scenario_type",
    "accuracy", "balanced_accuracy", "precision_attack", "recall_attack", "f1_attack",
]


def label_performance_cases(rows, case_type, selection_rule):
    output = rows[performance_case_columns].copy()
    output.insert(0, "selection_rule", selection_rule)
    output.insert(0, "case_type", case_type)
    return output


evaluation_with_gap = evaluation.assign(accuracy_f1_attack_gap=evaluation["accuracy"] - evaluation["f1_attack"])
best_per_scenario_type = (
    evaluation.sort_values(
        ["scenario_type", "f1_attack", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("scenario_type", as_index=False)
    .head(2)
)
worst_per_scenario_type = (
    evaluation.sort_values(
        ["scenario_type", "f1_attack", "balanced_accuracy"],
        ascending=[True, True, True],
    )
    .groupby("scenario_type", as_index=False)
    .head(2)
)
performance_cases = pd.concat([
    label_performance_cases(
        best_per_scenario_type,
        "melhor_por_tipo_de_cenario",
        "F1 de ataque decrescente e acurácia balanceada como desempate",
    ),
    label_performance_cases(
        worst_per_scenario_type,
        "pior_por_tipo_de_cenario",
        "F1 de ataque crescente e acurácia balanceada como desempate",
    ),
    label_performance_cases(
        evaluation_with_gap.nlargest(5, "accuracy_f1_attack_gap"),
        "acuracia_alta_e_f1_ataque_baixo",
        "cinco maiores valores de accuracy menos f1_attack",
    ),
    label_performance_cases(
        evaluation[evaluation["scenario_type"].eq("combined_to_unseen")].nlargest(5, "f1_attack"),
        "melhor_combinacao_para_base_nao_vista",
        "cinco maiores F1 de ataque no cenário combined_to_unseen",
    ),
], ignore_index=True)

save_table(comparison_cases, COMPARISON_TABLES_DIR, "cases_baseline_vs_randomized_search.csv")
save_table(performance_cases, COMPARISON_TABLES_DIR, "cases_performance_highlights.csv")
display(comparison_cases)
display(performance_cases)

,case_type,selection_rule,model,training_source,test_source,scenario_type,f1_attack_baseline,f1_attack_randomized_search,f1_attack_delta,balanced_accuracy_delta,precision_attack_delta,recall_attack_delta
0,maior_melhoria_f1_ataque,três maiores valores de f1_attack_delta,linear_svm,cicids2017,iot23,cross_dataset,0.044105,0.358254,0.314148,0.444643,0.190306,0.893584
1,maior_melhoria_f1_ataque,três maiores valores de f1_attack_delta,linear_svm,cicids2017,cicids2017,intra_dataset,0.664926,0.848853,0.183928,0.179632,-0.069519,0.406738
2,maior_melhoria_f1_ataque,três maiores valores de f1_attack_delta,linear_svm,cicids2017_iot23,unsw_nb15,combined_to_unseen,0.025785,0.156293,0.130508,0.035794,0.261059,0.084270
3,maior_piora_f1_ataque,três menores valores de f1_attack_delta,decision_tree,iot23,cicids2017,cross_dataset,0.550507,0.025045,-0.525462,-0.233397,-0.523883,-0.501263
4,maior_piora_f1_ataque,três menores valores de f1_attack_delta,mlp,unsw_nb15,cicids2017,cross_dataset,0.450475,0.332993,-0.117482,-0.095163,-0.177978,0.019173
5,maior_piora_f1_ataque,três menores valores de f1_attack_delta,mlp,iot23,unsw_nb15,cross_dataset,0.119335,0.012886,-0.106449,0.009537,0.260111,-0.083478
6,menor_alteracao_f1_ataque,três menores variações absolutas de f1_attack_delta,random_forest,cicids2017,unsw_nb15,cross_dataset,0.000000,0.000000,0.000000,0.000056,0.000000,0.000000
7,menor_alteracao_f1_ataque,três menores variações absolutas de f1_attack_delta,xgboost,cicids2017,unsw_nb15,cross_dataset,0.000000,0.000000,0.000000,0.000043,0.000000,0.000000
8,menor_alteracao_f1_ataque,três menores variações absolutas de f1_attack_delta,linear_svm,iot23,iot23,intra_dataset,0.993126,0.993102,-0.000025,-0.000006,-0.000049,0.000000
9,maior_melhoria_precisao_ataque,três maiores valores de precision_attack_delta,random_forest,iot23,unsw_nb15,cross_dataset,0.177697,0.121307,-0.056390,0.018039,0.284894,-0.080753


,case_type,selection_rule,model,training_method,training_source,test_source,scenario_type,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack
0,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,mlp,baseline,unsw_nb15_iot23,cicids2017,combined_to_unseen,0.800436,0.714051,0.500958,0.570077,0.533287
1,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,mlp,randomized_search,unsw_nb15_iot23,cicids2017,combined_to_unseen,0.790706,0.703233,0.479993,0.557444,0.515827
2,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,linear_svm,baseline,unsw_nb15,cicids2017,cross_dataset,0.842596,0.707462,0.641703,0.482239,0.550659
3,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,decision_tree,baseline,iot23,cicids2017,cross_dataset,0.831221,0.713302,0.588956,0.516770,0.550507
4,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,xgboost,baseline,iot23,iot23,intra_dataset,0.998851,0.997238,0.999701,0.994550,0.997119
5,melhor_por_tipo_de_cenario,F1 de ataque decrescente e acurácia balanceada como desempate,random_forest,randomized_search,iot23,iot23,intra_dataset,0.998841,0.997232,0.999651,0.994550,0.997094
6,pior_por_tipo_de_cenario,F1 de ataque crescente e acurácia balanceada como desempate,xgboost,baseline,cicids2017_iot23,unsw_nb15,combined_to_unseen,0.799088,0.499579,0.074074,0.000396,0.000788
7,pior_por_tipo_de_cenario,F1 de ataque crescente e acurácia balanceada como desempate,xgboost,randomized_search,cicids2017_iot23,unsw_nb15,combined_to_unseen,0.799832,0.500062,0.257143,0.000446,0.000890
8,pior_por_tipo_de_cenario,F1 de ataque crescente e acurácia balanceada como desempate,mlp,randomized_search,iot23,cicids2017,cross_dataset,0.786852,0.491782,0.000000,0.000000,0.000000
9,pior_por_tipo_de_cenario,F1 de ataque crescente e acurácia balanceada como desempate,xgboost,baseline,cicids2017,unsw_nb15,cross_dataset,0.799713,0.499820,0.000000,0.000000,0.000000


In [5]:
# gráficos Plotly de qualidade média por algoritmo e tipo de cenário
def plot_metric_by_scenario(metric, metric_label, filename):
    plot_data = evaluation.groupby(
        ["scenario_type", "model", "training_method"], as_index=False
    )[metric].mean()
    plot_data["model_label"] = plot_data["model"].map(MODEL_LABELS)
    plot_data["method_label"] = plot_data["training_method"].map(METHOD_LABELS)
    plot_data["scenario_label"] = plot_data["scenario_type"].map(SCENARIO_LABELS)
    figure = px.bar(
        plot_data,
        x="model_label",
        y=metric,
        color="method_label",
        facet_col="scenario_label",
        barmode="group",
        color_discrete_map=METHOD_COLORS,
        category_orders={
            "model_label": [MODEL_LABELS[model] for model in MODEL_ORDER],
            "method_label": [METHOD_LABELS[method] for method in METHOD_ORDER],
            "scenario_label": [SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
        },
        labels={"model_label": "Algoritmo", metric: metric_label, "method_label": "Método"},
        title=f"{metric_label} médio por algoritmo e tipo de cenário",
    )
    figure.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
    figure.update_yaxes(range=[0, 1.05], matches="y")
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1500, height=520)


plot_metric_by_scenario("f1_attack", "F1 da classe ATTACK", "f1_attack_by_scenario_type")
plot_metric_by_scenario("balanced_accuracy", "Acurácia balanceada", "balanced_accuracy_by_scenario_type")
plot_metric_by_scenario("precision_attack", "Precisão da classe ATTACK", "precision_attack_by_scenario_type")

In [6]:
# médias e desvios-padrão dos cinco folds em gráficos Plotly
def plot_cv_metric(metric, metric_label, filename):
    plot_data = cv_results.copy()
    plot_data["model_label"] = plot_data["model"].map(MODEL_LABELS)
    plot_data["method_label"] = plot_data["training_method"].map(METHOD_LABELS)
    plot_data["training_source_label"] = plot_data["training_source"].map(SOURCE_LABELS)
    figure = px.scatter(
        plot_data,
        x="model_label",
        y=f"{metric}_mean",
        error_y=f"{metric}_std",
        color="method_label",
        facet_col="training_source_label",
        facet_col_wrap=2,
        color_discrete_map=METHOD_COLORS,
        category_orders={
            "model_label": [MODEL_LABELS[model] for model in MODEL_ORDER],
            "method_label": [METHOD_LABELS[method] for method in METHOD_ORDER],
            "training_source_label": [SOURCE_LABELS[source] for source in TRAINING_SOURCE_ORDER],
        },
        labels={"model_label": "Algoritmo", f"{metric}_mean": metric_label, "method_label": "Método"},
        title=f"Validação cruzada: {metric_label} médio e desvio-padrão",
    )
    figure.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
    figure.update_traces(marker=dict(size=9))
    figure.update_yaxes(range=[0, 1.05], matches="y")
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1350, height=1250)


plot_cv_metric("f1_attack", "F1 da classe ATTACK", "cv_f1_attack_mean_std")
plot_cv_metric("balanced_accuracy", "Acurácia balanceada", "cv_balanced_accuracy_mean_std")

In [7]:
# heatmaps Plotly de generalização separados por método
def plot_generalization_heatmap(method):
    pivot = (
        evaluation[evaluation["training_method"].eq(method)]
        .groupby(["training_source", "test_source"])["f1_attack"]
        .mean()
        .unstack("test_source")
        .reindex(index=TRAINING_SOURCE_ORDER, columns=TEST_SOURCE_ORDER)
        .rename(index=SOURCE_LABELS, columns=SOURCE_LABELS)
    )
    figure = px.imshow(
        pivot,
        text_auto=".3f",
        color_continuous_scale="YlGnBu",
        zmin=0,
        zmax=1,
        aspect="auto",
        labels={"x": "Dataset de teste", "y": "Origem do treinamento", "color": "F1 médio da classe ATTACK"},
        title=f"Generalização média entre datasets — {METHOD_LABELS[method]}",
    )
    finalize_figure(figure, f"generalization_f1_attack_{method}", width=900, height=720)


for method in METHOD_ORDER:
    plot_generalization_heatmap(method)

delta_pivot = (
    comparison.groupby(["training_source", "test_source"])["f1_attack_delta"]
    .mean()
    .unstack("test_source")
    .reindex(index=TRAINING_SOURCE_ORDER, columns=TEST_SOURCE_ORDER)
    .rename(index=SOURCE_LABELS, columns=SOURCE_LABELS)
)
delta_limit = max(float(np.nanmax(np.abs(delta_pivot.to_numpy(dtype=float)))), 0.01)
delta_figure = px.imshow(
    delta_pivot,
    text_auto=".3f",
    color_continuous_scale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Dataset de teste", "y": "Origem do treinamento", "color": "Variação do F1 da classe ATTACK"},
    title="Variação média do F1 de ataque após o RandomizedSearchCV",
)
finalize_figure(delta_figure, "generalization_f1_attack_delta_heatmap", width=900, height=720)

WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/generalization_f1_attack_delta_heatmap.svg')

In [8]:
# comparação Plotly de custo, tamanho, vazão e variação do F1 de ataque
training_unique = evaluation.drop_duplicates(["model", "training_method", "training_source"])
efficiency_summary = (
    training_unique.groupby(["model", "training_method"], as_index=False)
    .agg(
        total_training_minutes=("training_elapsed_minutes", "sum"),
        mean_model_size_mb=("model_size_mb", "mean"),
    )
)
throughput_summary = (
    evaluation.groupby(["model", "training_method"], as_index=False)
    .agg(median_prediction_rows_per_second=("prediction_rows_per_second", "median"))
)
efficiency_summary = efficiency_summary.merge(throughput_summary, on=["model", "training_method"], how="left")
save_table(efficiency_summary, COMPARISON_TABLES_DIR, "efficiency_summary_by_model.csv")

efficiency_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Tempo total de treinamento (min)",
        "Tamanho médio do modelo (MB)",
        "Mediana de predições por segundo",
    ],
)
efficiency_metrics = ["total_training_minutes", "mean_model_size_mb", "median_prediction_rows_per_second"]
for column_index, metric in enumerate(efficiency_metrics, start=1):
    for method in METHOD_ORDER:
        method_label = METHOD_LABELS[method]
        values = (
            efficiency_summary[efficiency_summary["training_method"].eq(method)]
            .set_index("model")[metric]
            .reindex(MODEL_ORDER)
        )
        efficiency_figure.add_trace(
            go.Bar(
                x=[MODEL_LABELS[model] for model in MODEL_ORDER],
                y=values,
                name=method_label,
                marker_color=METHOD_COLORS[method_label],
                legendgroup=method_label,
                showlegend=column_index == 1,
                offsetgroup=method_label,
            ),
            row=1,
            col=column_index,
        )
efficiency_figure.update_layout(barmode="group", title="Custo e eficiência dos modelos")
efficiency_figure.update_xaxes(tickangle=-25, title_text="Algoritmo")
finalize_figure(efficiency_figure, "model_cost_and_efficiency", width=1500, height=560)

delta_extremes = pd.concat([
    comparison.nsmallest(8, "f1_attack_delta"),
    comparison.nlargest(8, "f1_attack_delta"),
], ignore_index=True).drop_duplicates(["model", "training_source", "test_source"])
delta_extremes = delta_extremes.sort_values("f1_attack_delta")
delta_extremes["case_label"] = delta_extremes.apply(
    lambda row: (
        f"{MODEL_LABELS[row['model']]} | {SOURCE_LABELS[row['training_source']]} → {SOURCE_LABELS[row['test_source']]}"
    ),
    axis=1,
)
delta_extremes["direction"] = np.where(delta_extremes["f1_attack_delta"].ge(0), "Melhoria", "Piora")
delta_figure = px.bar(
    delta_extremes,
    x="f1_attack_delta",
    y="case_label",
    color="direction",
    orientation="h",
    color_discrete_map={"Melhoria": "#2E7D32", "Piora": "#C62828"},
    labels={"f1_attack_delta": "Variação do F1 da classe ATTACK", "case_label": "Modelo e cenário", "direction": "Resultado"},
    title="Maiores melhorias e pioras após o RandomizedSearchCV",
)
delta_figure.add_vline(x=0, line_width=1, line_color="#444444")
finalize_figure(delta_figure, "f1_attack_largest_changes", width=1200, height=850)

WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/f1_attack_largest_changes.svg')

In [9]:
# resumo dos artefatos gerados e rankings preliminares
generated_tables = sorted(path.relative_to(TABLES_DIR).as_posix() for path in TABLES_DIR.rglob("*.csv"))
generated_figures = sorted(path.name for path in FIGURES_DIR.glob("*.svg"))
print(f"Tabelas salvas: {len(generated_tables)}")
print(f"Gráficos salvos em SVG: {len(generated_figures)}")
display(pd.DataFrame({"tabelas": generated_tables}))
display(pd.DataFrame({"graficos_svg": generated_figures}))

# o ranking usa F1 de ataque como critério principal e acurácia balanceada como desempate
global_ranking = evaluation.sort_values(
    ["f1_attack", "balanced_accuracy"],
    ascending=[False, False],
)[performance_case_columns]
display(global_ranking.head(10))
display(global_ranking.tail(10))

Tabelas salvas: 23
Gráficos salvos em SVG: 10


,tabelas
0,baseline/best_model_by_scenario.csv
1,baseline/final_metrics_all_scenarios.csv
2,baseline/kfold_metrics.csv
3,baseline/prediction_time_and_throughput.csv
4,baseline/scenario_combined_to_unseen_all_experiments.csv
5,baseline/scenario_cross_dataset_all_experiments.csv
6,baseline/scenario_intra_dataset_all_experiments.csv
7,baseline/training_cost_and_model_size.csv
8,comparisons/baseline_vs_randomized_search_all_scenarios.csv
9,comparisons/baseline_vs_randomized_search_combined_to_unseen.csv


,graficos_svg
0,balanced_accuracy_by_scenario_type.svg
1,cv_balanced_accuracy_mean_std.svg
2,cv_f1_attack_mean_std.svg
3,f1_attack_by_scenario_type.svg
4,f1_attack_largest_changes.svg
5,generalization_f1_attack_baseline.svg
6,generalization_f1_attack_delta_heatmap.svg
7,generalization_f1_attack_randomized_search.svg
8,model_cost_and_efficiency.svg
9,precision_attack_by_scenario_type.svg


,model,training_method,training_source,test_source,scenario_type,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack
68,xgboost,baseline,iot23,iot23,intra_dataset,0.998851,0.997238,0.999701,0.994550,0.997119
67,random_forest,randomized_search,iot23,iot23,intra_dataset,0.998841,0.997232,0.999651,0.994550,0.997094
69,xgboost,randomized_search,iot23,iot23,intra_dataset,0.998831,0.997207,0.999651,0.994501,0.997069
61,decision_tree,randomized_search,iot23,iot23,intra_dataset,0.998821,0.997164,0.999701,0.994402,0.997044
60,decision_tree,baseline,iot23,iot23,intra_dataset,0.998682,0.997207,0.998657,0.994749,0.996699
66,random_forest,baseline,iot23,iot23,intra_dataset,0.998662,0.997176,0.998607,0.994699,0.996649
64,mlp,baseline,iot23,iot23,intra_dataset,0.998365,0.996006,0.999750,0.992073,0.995897
65,mlp,randomized_search,iot23,iot23,intra_dataset,0.998147,0.995424,0.999850,0.990884,0.995347
62,linear_svm,baseline,iot23,iot23,intra_dataset,0.997255,0.995052,0.994879,0.991380,0.993126
63,linear_svm,randomized_search,iot23,iot23,intra_dataset,0.997245,0.995046,0.994830,0.991380,0.993102


,model,training_method,training_source,test_source,scenario_type,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack
25,mlp,randomized_search,cicids2017,unsw_nb15,cross_dataset,0.797860,0.499053,0.081395,0.001040,0.002054
24,mlp,baseline,cicids2017,unsw_nb15,cross_dataset,0.799514,0.499994,0.197531,0.000793,0.001579
21,decision_tree,randomized_search,cicids2017,unsw_nb15,cross_dataset,0.776894,0.485782,0.005093,0.000595,0.001065
39,xgboost,randomized_search,cicids2017_iot23,unsw_nb15,combined_to_unseen,0.799832,0.500062,0.257143,0.000446,0.000890
38,xgboost,baseline,cicids2017_iot23,unsw_nb15,combined_to_unseen,0.799088,0.499579,0.074074,0.000396,0.000788
27,random_forest,randomized_search,cicids2017,unsw_nb15,cross_dataset,0.799950,0.499969,0.000000,0.000000,0.000000
26,random_forest,baseline,cicids2017,unsw_nb15,cross_dataset,0.799861,0.499913,0.000000,0.000000,0.000000
29,xgboost,randomized_search,cicids2017,unsw_nb15,cross_dataset,0.799782,0.499864,0.000000,0.000000,0.000000
28,xgboost,baseline,cicids2017,unsw_nb15,cross_dataset,0.799713,0.499820,0.000000,0.000000,0.000000
55,mlp,randomized_search,iot23,cicids2017,cross_dataset,0.786852,0.491782,0.000000,0.000000,0.000000
